# Pretrained and from-scratch training

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SilenceX12138/TabFORGE/blob/master/docs/tutorial/notebooks/generation/02_pretrained_and_scratch_training.ipynb)

> **Colab setup:** Open this notebook with the badge, then follow the [tutorial setup guide](https://silencex12138.github.io/TabFORGE/#/tutorials.md) to install `tabforge-ml[tutorials]` from PyPI in this runtime. Select a GPU for pretrained workloads.

## 1. Overview

The recommended path fine-tunes the Hugging Face-hosted `tabforge-v1` backbone. This tutorial also shows how to
opt into fresh Denoising-aligned Latent Decoder and diffusion weights for controlled research comparisons.

In [ ]:
%pip install "tabforge-ml[tutorials]"

## 2. Imports and setup

Scratch training stays disabled unless `TABFORGE_RUN_SCRATCH=1` is set.

In [ ]:
import os
import tempfile
from pathlib import Path

from tabforge import (
    MODEL_ID,
    TabFORGEEmbeddingConfig,
    TabFORGERuntimeConfig,
    TabFORGETrainingConfig,
)

RANDOM_STATE = 7
MAX_TRAIN_ROWS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_ROWS", "256"))
MAX_STEPS = int(os.environ.get("TABFORGE_TUTORIAL_MAX_STEPS", "10"))
DEVICE = os.environ.get("TABFORGE_TUTORIAL_DEVICE", "auto")

assert MODEL_ID == "tabforge-v1"

RUN_SCRATCH = os.environ.get("TABFORGE_RUN_SCRATCH", "0") == "1"

## 3. Prepare a real-world dataset

`TabularDataset` downloads German Credit from OpenML, keeps the mixed numerical/categorical schema,
and performs a reproducible stratified split. The tutorial caps the training rows to keep execution
short; increase `TABFORGE_TUTORIAL_MAX_ROWS` for a larger run.

In [ ]:
from tabcamel.data.dataset import TabularDataset

dataset = TabularDataset("credit-g", task_type="classification")
split = dataset.split("stratified", train_size=0.8, random_state=RANDOM_STATE)
train_set = split["train_set"]
test_set = split["test_set"]

X_train = train_set.X_df.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
y_train = train_set.y_s.iloc[:MAX_TRAIN_ROWS].reset_index(drop=True)
X_test = test_set.X_df.reset_index(drop=True)
y_test = test_set.y_s.reset_index(drop=True)
categorical_features = tuple(dataset.categorical_feature_list)

print({"train": X_train.shape, "test": X_test.shape, "target": y_train.name})

## 4. Configure TabFORGE

Both paths share the same data, architecture, runtime, and optimizer budget.

In [ ]:
from tabforge import TabFORGEGenerator

def build_generator() -> TabFORGEGenerator:
    return TabFORGEGenerator(
        task="classification",
        categorical_features=categorical_features,
        embedding_config=TabFORGEEmbeddingConfig(model_path="auto", n_folds=2),
        training_config=TabFORGETrainingConfig(max_steps=MAX_STEPS, batch_size=128),
        runtime_config=TabFORGERuntimeConfig(device=DEVICE),
        random_state=RANDOM_STATE,
    )

## 5. Fit the recommended pretrained path

This is the default release workflow.

In [ ]:
pretrained_model = build_generator().fit(
    X_train,
    y_train,
    checkpoint="pretrained",
)

## 6. Generate

Generate a small table to verify the fitted pretrained path.

In [ ]:
pretrained_features, pretrained_target = pretrained_model.generate(16, random_state=RANDOM_STATE + 1)
print({"generated_features": pretrained_features.shape, "generated_targets": pretrained_target.shape})

## 7. Optional scratch comparison

Passing `checkpoint=None` initializes the TabFORGE Denoising-aligned Latent Decoder and diffusion Transformer from scratch. It
still uses pretrained Structure-aware Feature Encoder embeddings and generally requires a much larger budget.

In [ ]:
scratch_model = None
if RUN_SCRATCH:
    scratch_model = build_generator().fit(X_train, y_train, checkpoint=None)
    scratch_sample = scratch_model.generate(16, random_state=RANDOM_STATE + 1)
    print({"scratch_features": scratch_sample[0].shape})
else:
    print("Scratch run skipped; set TABFORGE_RUN_SCRATCH=1 to enable it.")

## 8. Save and load

Save the recommended fitted model as one canonical checkpoint.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    checkpoint_path = pretrained_model.save_checkpoint(Path(directory))
    restored = TabFORGEGenerator.restore_from_checkpoint(checkpoint_path, device=DEVICE)
    restored_sample = restored.generate(4, random_state=RANDOM_STATE + 2)

print({"checkpoint": checkpoint_path.name, "restored_rows": len(restored_sample[0])})

## 9. Optional advanced configuration

For a fair pretrained-versus-scratch experiment, hold the data split, architecture, optimizer steps,
masking policy, sampling seed, and evaluator fixed. Scratch runs need an independently justified budget.